# Meta cells with SEACells

Bolero is trained on **metacells** — small, homogeneous groups of cells pooled into
pseudobulks — rather than sparse single cells. This page takes the
[cell embedding](01_cell_embedding.ipynb) from the previous step, assigns metacells with
**SEACells**, and checks their quality.


## Setup

In [ ]:
from pathlib import Path
from warnings import simplefilter

import anndata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from bolero.tl.pseudobulk import prepare_multi_level_categorical_groups, run_meta_cells
from bolerodata import DATASETS

simplefilter("ignore")
sc.set_figure_params(figsize=(4, 4), frameon=False)

In [ ]:
# --- Configuration ---------------------------------------------------------------
DATASET_NAME = "ChromiumPBMC"
WORK_DIR = Path(
    "/large_storage/zhoulab/hanliu/wmb/standard/embedding/poissonmultivi/ChromiumPBMC"
)

# Metacells are called separately within each "limit group", so cells of different
# broad types are never pooled together.
META_CELL_LIMIT = ["subclass"]
MAX_FRAGMENTS = 3_000_000     # cap total fragments per metacell
META_FOLD = 75                # ~ target number of cells per metacell
MIN_CELLS = 200               # skip / merge groups smaller than this

dataset = DATASETS[DATASET_NAME]

## Step 1 — Load the embedding and define limit groups

`prepare_multi_level_categorical_groups` walks the categorical columns in `META_CELL_LIMIT`
and uses the embedding to merge categories that are too small to stand on their own,
producing one `group` label per cell. Metacells are then built *within* each group.

In [ ]:
cell_meta = dataset.make_cell_metadata().dropna(subset=META_CELL_LIMIT)
embedding = pd.read_feather(WORK_DIR / "pmvi.latent_embedding.feather")

limit_group, group_to_categories = prepare_multi_level_categorical_groups(
    cell_meta, embedding, META_CELL_LIMIT, min_cell_count=MIN_CELLS
)
group_to_categories

In [ ]:
adata = anndata.read_h5ad(WORK_DIR / "adata.pmvi.with_coords.h5ad")
adata.obs["group"] = limit_group
adata.obs["group"].value_counts()

## Step 2 — Call metacells

`run_meta_cells` runs SEACells within each group, splitting very large groups and enforcing
the fragment cap. It returns a per-cell metacell assignment, which we store on the `AnnData`
and to disk.

In [ ]:
meta_cell_assign = run_meta_cells(
    adata,
    obsm="X_pmvi",
    groupby="group",
    max_fragments=MAX_FRAGMENTS,
    meta_fold=META_FOLD,
    group_size_cutoff=MIN_CELLS,
)
adata.obs["meta_cell"] = meta_cell_assign
meta_cell_assign.to_csv(WORK_DIR / "seacell.assign.csv.gz")
adata.write_h5ad(WORK_DIR / "adata.pmvi.with_coords.h5ad")

## Step 3 — Visualize metacells on the UMAP

In [ ]:
umap = pd.DataFrame(adata.obsm["X_umap"], index=adata.obs_names, columns=["x", "y"])
codes = adata.obs["meta_cell"].astype("category").cat.codes

fig, axes = plt.subplots(figsize=(9, 4), ncols=2, dpi=150, constrained_layout=True)
axes[0].scatter(umap["x"], umap["y"], s=4, c=codes % 20, cmap="tab20")
axes[0].set_title("metacell assignment")
axes[0].axis("off")

# one representative cell per metacell, in red over a grey background
centers = umap.groupby(adata.obs["meta_cell"], observed=True).sample(1, random_state=0)
axes[1].scatter(umap["x"], umap["y"], s=2, color="lightgrey")
axes[1].scatter(centers["x"], centers["y"], s=8, color="red")
axes[1].set_title("metacell centers")
axes[1].axis("off")

### Metacell coverage

Pooling lifts coverage from single-cell sparsity to usable pseudobulk depth. These histograms
summarize per-metacell fragment / UMI totals and mean quality.

In [ ]:
cell_meta = cell_meta.reindex(adata.obs_names)
cell_meta["meta_cell"] = adata.obs["meta_cell"]

n_cell = cell_meta["meta_cell"].notna().sum()
n_meta = cell_meta["meta_cell"].dropna().nunique()
print(f"{n_meta} metacells from {n_cell} cells")

meta_cov = cell_meta.groupby("meta_cell").agg(
    {"n_fragments": "sum", "tsse": "mean", "n_umis": "sum", "n_genes": "mean"}
)
fig, axes = plt.subplots(figsize=(6, 3), nrows=2, ncols=2, dpi=150, constrained_layout=True)
for ax, col in zip(axes.ravel(), meta_cov.columns):
    sns.histplot(meta_cov[col], bins=50, ax=ax)

### Metacell purity

A good metacell is dominated by a single category of each variable. For every categorical
metadata column we plot the fraction of each metacell's cells that fall in its majority
category — values near 1 mean clean, homogeneous metacells.

In [ ]:
purity = {}
for col, data in cell_meta.items():
    if data.dtype.name != "category" or col == "meta_cell":
        continue
    counts = data.groupby(cell_meta["meta_cell"]).value_counts().unstack()
    frac = counts.div(counts.sum(axis=1), axis=0)
    purity[col] = frac.max(axis=1)
purity = pd.DataFrame(purity)

plot_cols = [
    c
    for c in ["sample", "age", "sex", "donor", "tissue", "DissectionRegion", "class", "subclass"]
    if c in purity
]
fig, axes = plt.subplots(
    figsize=(1.4 * len(plot_cols), 3), ncols=len(plot_cols), dpi=150, constrained_layout=True
)
for i, (ax, col) in enumerate(zip(np.atleast_1d(axes), plot_cols)):
    sns.boxplot(y=purity[col], ax=ax)
    ax.set(ylim=(0, 1), title=col, xlabel="", ylabel="purity" if i == 0 else "")
    if i:
        ax.set(yticklabels=[])
    ax.grid(axis="y")

---

`seacell.assign.csv.gz` (and the `meta_cell` column on the embedding `AnnData`) now map every
cell to a metacell. These metacells are the units Bolero aggregates into conditioned
pseudobulks for training and inference.